# 03 - Evaluating Model E on the test set

Evaluates the model saved by notebook 02 on the test molecules (`data/processed/test.csv`), which it has never seen.

Two levels of evaluation:
- **row by row** (molecule x HPLC column x solvent), as Arthur did: accuracy, precision, recall;
- **per molecule**: which solvent does the model suggest first, and was that choice right? This is how the model is actually used.

Each score is compared with a "no intelligence" rule: always answer "soluble", or always suggest DMSO.

In [1]:
import sys
from pathlib import Path
# Find the project root (the folder that contains src/config.py) and make src/ importable
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import TEST, MODEL_E

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [2]:
# The 16 features of Model E, in Arthur's order (cell 18): the order expected by scripts/predict.py
cols_E = ['MolWt', 'Sol_Dielectric', 'Sol_Hansen_D', 'Sol_Hansen_P', 'Sol_Hansen_H']
vector_cols = [f'Matrix_Vector_{i}' for i in range(1, 11)]
xgb_cols = ['RT'] + cols_E + vector_cols

def rank_per_molecule(df, proba):
    """One row per molecule x solvent: mean predicted probability over the molecule's HPLC columns."""
    g = df[['SMILES', 'Solvent_Name', 'Soluble']].assign(proba=proba)
    return g.groupby(['SMILES', 'Solvent_Name'], as_index=False).agg(proba=('proba', 'mean'), Soluble=('Soluble', 'first'))

def first_choices(ranking):
    """For each molecule: the solvent ranked first, and whether that choice was right (the molecule dissolves in it)."""
    rows = []
    for smiles, g in ranking.groupby('SMILES'):
        best = g.loc[g['proba'].idxmax()]
        dmso = g[g['Solvent_Name'] == 'DMSO']
        rows.append({'SMILES': smiles, 'Solvent': best['Solvent_Name'], 'Correct': best['Soluble'] == 1,
                     'DMSO_correct': len(dmso) > 0 and dmso['Soluble'].iloc[0] == 1})
    return pd.DataFrame(rows)

In [3]:
df_test = pd.read_csv(TEST)
model_E = joblib.load(MODEL_E)
print(f"Test: {len(df_test)} rows, {df_test['SMILES'].nunique()} molecules | model: {MODEL_E.name}")

Test: 1670 rows, 59 molecules | model: xgb_model_e_16features.joblib


## Row-by-row scores (as Arthur)

In [4]:
X_test_df = df_test[xgb_cols]
y_test = df_test['Soluble'].values
y_pred_E = model_E.predict(X_test_df)

print(f"Accuracy:  {accuracy_score(y_test, y_pred_E):.4f}   (always 'soluble': {np.mean(y_test == 1):.4f})")
print(f"Precision: {precision_score(y_test, y_pred_E, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_E, zero_division=0):.4f}")

Accuracy:  0.6611   (always 'soluble': 0.7006)
Precision: 0.7838
Recall:    0.7128


### Per solvent

In [5]:
per_solvent = []
for solvent, g in df_test.assign(pred=y_pred_E).groupby('Solvent_Name'):
    per_solvent.append({'Solvent': solvent, 'Rows': len(g),
                        'Accuracy': accuracy_score(g['Soluble'], g['pred']),
                        'Precision': precision_score(g['Soluble'], g['pred'], zero_division=0),
                        'Recall': recall_score(g['Soluble'], g['pred'], zero_division=0),
                        'Always soluble': np.mean(g['Soluble'] == 1)})
display(pd.DataFrame(per_solvent).set_index('Solvent').style.format("{:.3f}", subset=['Accuracy', 'Precision', 'Recall', 'Always soluble']))

,Rows,Accuracy,Precision,Recall,Always soluble
Solvent,,,,,
ACN,334,0.677,0.756,0.705,0.620
CHCl3,334,0.722,0.931,0.704,0.799
DCM,334,0.647,0.884,0.635,0.787
DMSO,334,0.662,0.725,0.823,0.692
MeOH,334,0.599,0.656,0.708,0.605


## Solvent choice, per molecule

For each test molecule, the model ranks the 5 solvents (mean of its probabilities over the molecule's HPLC columns). We look at the solvent ranked first.

In [6]:
ranking = rank_per_molecule(df_test, model_E.predict_proba(X_test_df)[:, 1])
choices = first_choices(ranking)
print(f"Correct solvent choice (top-1): {choices['Correct'].mean():.3f}   (always DMSO: {choices['DMSO_correct'].mean():.3f})")

order = ['MeOH', 'ACN', 'DMSO', 'DCM', 'CHCl3']
distribution = pd.DataFrame({
    'Molecules': choices['Solvent'].value_counts().reindex(order, fill_value=0),
    '% of cases': (choices['Solvent'].value_counts(normalize=True) * 100).reindex(order, fill_value=0).round(1),
    'Correct (%)': (choices.groupby('Solvent')['Correct'].mean() * 100).reindex(order).round(1),
})
display(distribution)

Correct solvent choice (top-1): 0.763   (always DMSO: 0.661)


,Molecules,% of cases,Correct (%)
Solvent,,,
MeOH,6,10.2,33.3
ACN,0,0.0,NaN
DMSO,29,49.2,69.0
DCM,1,1.7,100.0
CHCl3,23,39.0,95.7
